# Slice 2: Dataset Acquisition

This notebook downloads a pre-processed Deepfake dataset (Real vs Fake faces) to speed up training.

**Dataset used:** `xhlulu/140k-real-and-fake-faces` (Kaggle)

### Prerequisites:
1. You need a Kaggle account.
2. Create a new API Token: Go to Kaggle Settings > Account > Create New API Token -> downloads `kaggle.json`.
3. Upload `kaggle.json` to your Google Drive Project Folder (inside `deepfakedetection` root).

In [ ]:
from google.colab import drive
import os

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Navigate to Project Folder
# CHANGE THIS PATH IF NEEDED
project_path = '/content/drive/My Drive/deepfakedetection'
os.chdir(project_path)
print(f"Current working directory: {os.getcwd()}")

In [ ]:
# 3. Setup Kaggle Config
import os

# Check if kaggle.json exists in the project folder
if not os.path.exists("kaggle.json"):
    print("ERROR: kaggle.json not found in the project directory!")
    print("Please upload your kaggle.json file to: " + os.getcwd())
else:
    # Move/Copy logic specifically for Colab's kaggle expectations
    os.environ['KAGGLE_CONFIG_DIR'] = os.getcwd()
    # Secure the key
    !chmod 600 kaggle.json
    print("Kaggle configuration set.")

In [ ]:
# 4. Download Dataset
# splitting the download command to avoid redownloading if exists

if not os.path.exists("real_vs_fake"):
    print("Downloading dataset... (approx 3.7GB)")
    !kaggle datasets download -d xhlulu/140k-real-and-fake-faces
    
    print("Unzipping...")
    !unzip -q 140k-real-and-fake-faces.zip
    
    # Cleanup zip to save space
    !rm 140k-real-and-fake-faces.zip
    print("Download and Extraction Complete.")
else:
    print("Dataset folder 'real_vs_fake' already exists. Skipping download.")

In [ ]:
# 5. Reorganize into standard data/ structure
# The downloaded dataset usually has structure: real_vs_fake/real-vs-fake/...
# We want: data/real and data/fake

import shutil
from tqdm.notebook import tqdm

source_dir = "real_vs_fake/real-vs-fake"
target_dir = "data"

if not os.path.exists(target_dir):
    os.makedirs(target_dir)

categories = ["real", "fake"]

dataset_ready = True
for cat in categories:
    path = os.path.join(target_dir, cat)
    if not os.path.exists(path):
        dataset_ready = False
        os.makedirs(path)

if dataset_ready and len(os.listdir(f"{target_dir}/real")) > 0:
    print("Data already organized in data/ folder.")
else:
    print("Moving files to data/ folder...")
    # This specific dataset has train/test/valid split inside
    # We will merge them all into one big pool for our own custom splitting logic later
    # OR just map them directly. Let's merge for simplicity of the pipeline guide.
    
    subsets = ["train", "valid", "test"]
    
    for subset in subsets:
        for cat in categories:
            src_path = os.path.join(source_dir, subset, cat)
            dst_path = os.path.join(target_dir, cat)
            
            if os.path.exists(src_path):
                files = os.listdir(src_path)
                print(f"Moving {len(files)} {cat} images from {subset}...")
                for f in tqdm(files):
                    shutil.move(os.path.join(src_path, f), os.path.join(dst_path, f))
    
    print("Organization Complete.")

In [ ]:
# 6. Verify Counts
real_count = len(os.listdir("data/real"))
fake_count = len(os.listdir("data/fake"))

print(f"Final Dataset Stats:")
print(f"Real Images: {real_count}")
print(f"Fake Images: {fake_count}")

assert real_count > 0, "No real images found!"
assert fake_count > 0, "No fake images found!"
print("Slice 2 Verification Passed! ✅")